# Mini-Project: Loan Status Prediction
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

Binary classification on mixed categorical and numeric loan application data using SVM.

Main goals:

- Handle missing values across both numeric and categorical columns.
- Encode categorical features and scale numeric features.
- Use a stratified split and train an SVM classifier.
- Evaluate with the full metric set.

---

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

## Load the Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/mohitkhyalia1/sos_2026/refs/heads/main/dataset/loan_status_data.csv'

df = pd.read_csv(url)
print('Shape:', df.shape)
df.head()

## Missing Values

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
if 'Loan_ID' in categorical_cols:
    categorical_cols.remove('Loan_ID')
    df = df.drop('Loan_ID', axis=1)

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print('Missing values after filling:', df.isnull().sum().sum())

## EDA

In [ ]:
print(df['Loan_Status'].value_counts())
print(f'Approval rate: {(df["Loan_Status"]=="Y").mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x='Education', hue='Loan_Status', data=df, ax=axes[0],
              palette={'Y': '#2CA02C', 'N': '#C00000'}, alpha=0.85)
sns.countplot(x='Property_Area', hue='Loan_Status', data=df, ax=axes[1],
              palette={'Y': '#2CA02C', 'N': '#C00000'}, alpha=0.85)
plt.tight_layout()
plt.show()

print(df.groupby('Education')['Loan_Status'].apply(lambda x: (x=='Y').mean()).round(3))
print(df.groupby('Property_Area')['Loan_Status'].apply(lambda x: (x=='Y').mean()).round(3))

**Observation:**
Graduate applicants and Semiurban applicants both show a higher approval rate than the other categories within their respective columns.

## Encoding

In [ ]:
df_encoded = df.copy()
encoders = {}

for col in categorical_cols + ['Loan_Status']:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le

df_encoded.head()

## Train-Test Split and Scaling

In [ ]:
X = df_encoded.drop('Loan_Status', axis=1)
y = df_encoded['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}  Test: {X_test_scaled.shape}')
print(f'Train approval rate: {y_train.mean():.3f}  Test approval rate: {y_test.mean():.3f}')

## Training the SVM Classifier

In [ ]:
model = SVC(kernel='rbf', C=1)
model.fit(X_train_scaled, y_train)

train_preds = model.predict(X_train_scaled)
test_preds = model.predict(X_test_scaled)

print(f'Train accuracy: {accuracy_score(y_train, train_preds):.4f}')
print(f'Test accuracy:  {accuracy_score(y_test, test_preds):.4f}')

## Evaluation

In [ ]:
print(f'Precision: {precision_score(y_test, test_preds):.4f}')
print(f'Recall:    {recall_score(y_test, test_preds):.4f}')
print(f'F1:        {f1_score(y_test, test_preds):.4f}')
print()
print(classification_report(y_test, test_preds, target_names=['Rejected', 'Approved']))

In [ ]:
cm = confusion_matrix(y_test, test_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: Rejected', 'Pred: Approved'],
            yticklabels=['Actual: Rejected', 'Actual: Approved'],
            linewidths=0.5, linecolor='lightgray', cbar=False, annot_kws={'size': 13})
ax.set_title('Confusion Matrix — Loan Status')
plt.show()

**Observation:**
Most errors fall on the rejected class being predicted as approved — consistent with the approval-rate imbalance in the training data, the model leans toward the majority outcome at the decision boundary.

## Linear vs RBF Kernel Comparison

In [ ]:
for kernel in ['linear', 'rbf']:
    m = SVC(kernel=kernel, C=1).fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f'{kernel:8s} kernel test accuracy: {acc:.4f}')

**Observation:**
Linear and RBF perform similarly here, suggesting the decision boundary doesn't need much curvature to separate the classes on this feature set.